# 온전한 행동 선택 지시문 후속 결과
20261021T065748-ed168d74의 40라운드와 기존 30사례 실행 중 동일한 B/C 01~10 사례를 비교한다.
원본 JSONL과 공개 경험 파일만 읽으며 모델/임베딩/서버를 실행하지 않는다.
기존 대응군에는 오류 후 자동 행동을 포함한 실패 1회가 있다.


In [1]:
# Read-only comparison of the same 40 trials; no API or model requests.
from pathlib import Path
from collections import Counter, defaultdict
import json
import statistics

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/results").is_dir())
old_run = root / "experiments/results/20261021T041257-fab9483c"
new_run = root / "experiments/results/20261021T065748-ed168d74"
def read_run(path, stream):
    file = path / (stream + ".jsonl")
    return [json.loads(line) for line in file.read_text().splitlines() if line.strip()] if file.exists() else []
def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

new_trials = read_run(new_run, "trials")
ids = {r["trial_id"] for r in new_trials}
assert len(new_trials) == len(ids) == 40
assert all(r["valid"] for r in new_trials)
old_trials = {r["trial_id"]: r for r in read_run(old_run, "trials") if r["trial_id"] in ids}
assert len(old_trials) == 40
assert all(old_trials[r["trial_id"]]["input_hash"] == r["input_hash"] for r in new_trials)
old_manifest = json.loads((old_run / "manifest.json").read_text())
new_manifest = json.loads((new_run / "manifest.json").read_text())
assert old_manifest["model_digest"] == new_manifest["model_digest"]
show({"run": new_run.name, "status": new_manifest["status"], "valid_trials": 40,
      "seconds": new_manifest["total_duration_s"], "model": new_manifest["model"]})

comparisons = []
for path in (old_run, new_run):
    trials = [r for r in read_run(path, "trials") if r["trial_id"] in ids]
    events = [r for r in read_run(path, "events") if r["trial_id"] in ids]
    calls = [r for r in read_run(path, "model_calls") if r["trial_id"] in ids]
    requests = {r["call_id"]: r for r in calls if r["event"] == "request"}
    responses = [r for r in calls if r["event"] == "response"]
    assert len(requests) == len(responses)
    assert set(requests) == {r["call_id"] for r in responses}
    forced, invalid_json = Counter(), 0
    for response in responses:
        request = requests[response["call_id"]]
        user = json.loads(request["user"])
        try:
            answer = json.loads(response["content"])
        except ValueError:
            invalid_json += 1
            continue
        if "raise" in user["observation"]["allowed_bets"]:
            action = (answer.get("bet") or {}).get("action", "none")
            forced[f"force={user['force_bet']},action={action}"] += 1
    actions = Counter(r["event"]["action"] for r in events if r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A")
    auto = [r["event"] for r in events if r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A" and r["event"].get("automatic")]
    group_stats = []
    for opponent in ("B", "C"):
        for condition in ("control", "both"):
            group = [r for r in trials if r["opponent"] == opponent and r["condition"] == condition]
            group_stats.append({"opponent": opponent, "condition": condition, "valid": sum(r["valid"] for r in group),
                                "wins": sum(r["winner"] == "A" for r in group),
                                "mean_chip_delta": statistics.mean(r["chip_delta"] for r in group)})
    comparisons.append({"run": path.name, "cases": len(trials), "model_calls": len(requests),
                        "tokens": sum(r["usage"]["tokens"] for r in responses), "invalid_json_attempts": invalid_json,
                        "actions": dict(actions), "automatic_actions": [e["action"] for e in auto],
                        "forced_actions_raise_available": dict(forced), "groups": group_stats})
show(comparisons)
show({"winner_changes_old_vs_new": sum(old_trials[r["trial_id"]]["winner"] != r["winner"] for r in new_trials),
      "chip_delta_changes_old_vs_new": sum(old_trials[r["trial_id"]]["chip_delta"] != r["chip_delta"] for r in new_trials)})

# Verify the historical prompt and retrieved parent record were actually sent.
action_prompt = (root / "agents/prompts/poker_action_selection_standard.txt").read_text().strip()
new_requests = [r for r in read_run(new_run, "model_calls") if r["event"] == "request"]
game_requests = [r for r in new_requests if json.loads(r["user"])["observation"]["phase"] != "revealed"]
assert all(action_prompt in r["system"] for r in game_requests)
trial_map = {r["trial_id"]: r for r in new_trials}
attached = 0
for request in new_requests:
    for match in json.loads(request["user"]).get("past_experiences", {}).get("matches", []):
        context, meta = match["round_context"], match["metadata"]
        assert context["experience_id"] == meta["experience_id"]
        assert context["round_id"] == meta["round_id"]
        assert meta["player_id"] == trial_map[request["trial_id"]]["opponent"]
        archive = root / "experiments/results" / new_run.name / "data" / request["trial_id"] / "experiences" / (trial_map[request["trial_id"]]["game_id"] + ".json")
        stored = {r["id"]: r for r in json.loads(archive.read_text())}
        assert context["result"] == stored[meta["experience_id"]]["result"]
        attached += 1
show({"game_requests_with_new_prompt": len(game_requests), "retrieved_occurrences_with_verified_public_result": attached})
print("Input+output tokens include retries and post-round reflections. Original model errors include automatic actions.")
print("Prompt formatting and RAG also changed; this is not an isolated causal estimate for one instruction sentence.")


{
  "run": "20261021T065748-ed168d74",
  "status": "complete",
  "valid_trials": 40,
  "seconds": 2194.7620121670006,
  "model": "qwen3:8b"
}
[
  {
    "run": "20261021T041257-fab9483c",
    "cases": 40,
    "model_calls": 223,
    "tokens": 384881,
    "invalid_json_attempts": 12,
    "actions": {
      "raise": 45,
      "check": 1,
      "fold": 1
    },
    "automatic_actions": [
      "check",
      "fold"
    ],
    "forced_actions_raise_available": {
      "force=False,action=none": 62,
      "force=True,action=raise": 43,
      "force=False,action=raise": 2
    },
    "groups": [
      {
        "opponent": "B",
        "condition": "control",
        "valid": 9,
        "wins": 5,
        "mean_chip_delta": -0.6
      },
      {
        "opponent": "B",
        "condition": "both",
        "valid": 10,
        "wins": 5,
        "mean_chip_delta": -1
      },
      {
        "opponent": "C",
        "condition": "control",
        "valid": 10,
        "wins": 8,
        "mean_

In [2]:
# Explicit check diagnostic: compare raw model decisions with accepted engine actions.
from pathlib import Path
from collections import Counter
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/results").is_dir())
run = root / "experiments/results/20261021T074824-80533f94"
def diagnostic_rows(stream):
    path = run / (stream + ".jsonl")
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
trials = diagnostic_rows("trials")
calls = diagnostic_rows("model_calls")
requests = {r["call_id"]: r for r in calls if r["event"] == "request"}
responses = [r for r in calls if r["event"] == "response"]
assert len(requests) == len(responses) == 20
assert {r["call_id"] for r in responses} == set(requests)
game = [r for r in requests.values() if json.loads(r["user"])["observation"]["phase"] != "revealed"]
assert len(game) == 16
assert all("임시 체크 진단" in r["system"] for r in game)
force_true = sum(bool(json.loads(r["user"])["force_bet"]) for r in game)
assert force_true == 0
for request in game:
    user = json.loads(request["user"])
    if not user["observation"]["allowed_bets"]:
        continue
    allowed = {a for branch in request["schema"]["$defs"]["Bet"]["anyOf"] for a in branch["properties"]["action"]["enum"]}
    assert "raise" in allowed and ("check" in allowed or "call" in allowed)
events = diagnostic_rows("events")
actions = [r["event"] for r in events if r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A"]
assert all(not e.get("automatic") for e in actions)
result = {"run": run.name, "valid": sum(r["valid"] for r in trials), "attempted": len(trials),
          "model_calls": len(requests), "tokens": sum(r["usage"]["tokens"] for r in responses),
          "accepted_A_actions": dict(Counter(e["action"] for e in actions)), "force_bet_true_requests": force_true}
print(json.dumps(result, ensure_ascii=False, indent=2))
print("The existing grammar/engine accepted check and call. The force_bet=true path was not exercised.")


{
  "run": "20261021T074824-80533f94",
  "valid": 4,
  "attempted": 4,
  "model_calls": 20,
  "tokens": 39606,
  "accepted_A_actions": {
    "check": 4,
    "call": 2
  },
  "force_bet_true_requests": 0
}
The existing grammar/engine accepted check and call. The force_bet=true path was not exercised.


In [3]:
# Force-bet check diagnostic, immutable historical run; read-only, no model requests.
from pathlib import Path
from collections import Counter
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/results").is_dir())
run = root / "experiments/results/20261021T080638-0166cb81"
calls = [json.loads(line) for line in (run / "model_calls.jsonl").read_text().splitlines() if line.strip()]
requests = {r["call_id"]: r for r in calls if r["event"] == "request"}
responses = [r for r in calls if r["event"] == "response"]
assert len(requests) == len(responses) == 13
assert set(requests) == {r["call_id"] for r in responses}
choices = Counter()
for response in responses:
    request = requests[response["call_id"]]
    user = json.loads(request["user"])
    if user["observation"]["phase"] == "revealed":
        continue
    assert "force_bet=false이면 내 차례이고 allowed_bets가 있어도 항상 bet=null" in request["system"]
    decision = json.loads(response["content"])
    action = (decision.get("bet") or {}).get("action", "none")
    choices[f"force={user['force_bet']},action={action}"] += 1
    if user["force_bet"]:
        allowed = {a for branch in request["schema"]["$defs"]["Bet"]["anyOf"] for a in branch["properties"]["action"]["enum"]}
        assert "raise" in allowed and "fold" in allowed and action in allowed
    else:
        assert decision["bet"] is None
accepted = [r["event"] for r in map(json.loads, (run / "events.jsonl").read_text().splitlines())
            if r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A"]
assert all(not e.get("automatic") for e in accepted)
assert Counter(e["action"] for e in accepted) == {"check": 2, "call": 1}
print(json.dumps({"run": run.name, "choices": dict(choices), "accepted_actions": dict(Counter(e["action"] for e in accepted)),
                  "calls": len(requests), "tokens": sum(r["usage"]["tokens"] for r in responses)}, ensure_ascii=False, indent=2))


{
  "run": "20261021T080638-0166cb81",
  "choices": {
    "force=False,action=none": 8,
    "force=True,action=check": 2,
    "force=True,action=call": 1
  },
  "accepted_actions": {
    "check": 2,
    "call": 1
  },
  "calls": 13,
  "tokens": 23124
}


In [4]:
# OpenAI standard-prompt run: verify requested/reported model, actions and source grounding.
from pathlib import Path
from collections import Counter
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/results").is_dir())
run = root / "experiments/results/20261021T081635-76bb44b5"
def openai_rows(stream):
    return [json.loads(line) for line in (run / (stream + ".jsonl")).read_text().splitlines() if line.strip()]
manifest = json.loads((run / "manifest.json").read_text())
calls = openai_rows("model_calls")
requests = {r["call_id"]: r for r in calls if r["event"] == "request"}
responses = [r for r in calls if r["event"] == "response"]
assert len(requests) == len(responses) == 20
assert set(requests) == {r["call_id"] for r in responses}
assert {r["model"] for r in requests.values()} == {"openai:gpt-6-luna"}
assert {r["usage"]["actual_model"] for r in responses} == {"gpt-6-luna"}
assert manifest["backend"] == "openai"
assert manifest["model"] == manifest["provider_model_id"] == "gpt-6-luna"
assert manifest["source_hashes"]["agents/prompts/poker_action_selection.txt"] == manifest["source_hashes"]["agents/prompts/poker_action_selection_standard.txt"]
assert all(r["usage"]["tokens"] == r["usage"]["input_tokens"] + r["usage"]["output_tokens"] for r in responses)
trials = openai_rows("trials")
assert len(trials) == 4 and all(r["valid"] for r in trials)
accepted = [r["event"] for r in openai_rows("events") if r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A"]
assert all(not e.get("automatic") for e in accepted)
print(json.dumps({"run": run.name, "model": manifest["model"], "valid": len(trials), "seconds": manifest["total_duration_s"],
                  "calls": len(requests), "input_tokens": sum(r["usage"]["input_tokens"] for r in responses),
                  "output_tokens": sum(r["usage"]["output_tokens"] for r in responses),
                  "total_tokens": sum(r["usage"]["tokens"] for r in responses),
                  "A_actions": dict(Counter(e["action"] for e in accepted)),
                  "outcomes": [{k: r[k] for k in ("trial_id", "winner", "chip_delta")} for r in trials]}, ensure_ascii=False, indent=2))
# A claim about 'your card' from C refers to A, not C's own revealed card.
for request in requests.values():
    user = json.loads(request["user"])
    if request["trial_id"] != "s3-C-01-both" or user["observation"]["event_id"] != 3:
        continue
    matches = user["past_experiences"]["matches"]
    claims = [json.loads(m["text"].split("\n", 1)[1])["text"] for m in matches]
    ranks = [m["round_context"]["result"]["revealed_cards"]["A"]["rank"] for m in matches]
    assert claims == ["네 카드는 10 이하야."] * 3
    assert ranks == [7, 7, 4]
    response = next(r for r in responses if r["call_id"] == request["call_id"])
    print(json.dumps({"evidence_A_ranks": ranks, "true_claims": sum(rank <= 10 for rank in ranks),
                      "model_note": json.loads(response["content"])["memory_updates"]["round"]}, ensure_ascii=False, indent=2))


{
  "run": "20261021T081635-76bb44b5",
  "model": "gpt-6-luna",
  "valid": 4,
  "seconds": 58.93131558300229,
  "calls": 20,
  "input_tokens": 47967,
  "output_tokens": 2042,
  "total_tokens": 50009,
  "A_actions": {
    "check": 4,
    "fold": 1,
    "call": 1
  },
  "outcomes": [
    {
      "trial_id": "s3-B-01-both",
      "winner": "B",
      "chip_delta": -2
    },
    {
      "trial_id": "s3-B-01-control",
      "winner": "A",
      "chip_delta": 6
    },
    {
      "trial_id": "s3-C-01-both",
      "winner": "A",
      "chip_delta": 2
    },
    {
      "trial_id": "s3-C-01-control",
      "winner": "A",
      "chip_delta": 2
    }
  ]
}
{
  "evidence_A_ranks": [
    7,
    7,
    4
  ],
  "true_claims": 3,
  "model_note": "C의 ‘내 카드는 10 이하’ 주장은 과거 공개 결과 3건 중 1건만 일치해 신뢰하기 어렵다. 내 카드는 미확인이고 현재 추가 비용이 없어 체크를 잠정 선택한다."
}


In [5]:
from pathlib import Path
from collections import Counter
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/results").is_dir())
run_id = "20261021T083322-7fc60706"
run = root / "experiments/results" / run_id
def read_rows(name):
    return [json.loads(line) for line in (run / name).read_text().splitlines() if line.strip()]
manifest = json.loads((run / "manifest.json").read_text())
trials = read_rows("trials.jsonl")
calls = read_rows("model_calls.jsonl")
requests = {r["call_id"]: r for r in calls if r["event"] == "request"}
responses = [r for r in calls if r["event"] == "response"]
assert len(trials) == 4 and all(t["valid"] and not t["runner_errors"] for t in trials)
assert len(requests) == len(responses) == 20
assert all(r["model"] == "openai:gpt-6.1-sol" for r in requests.values())
assert all(r["usage"]["actual_model"] == "gpt-6.1-sol" for r in responses)
assert all(t["game_config"]["openai_reasoning_effort"] == "low" for t in trials)
assert manifest["source_hashes"]["agents/prompts/poker_action_selection.txt"] == manifest["source_hashes"]["agents/prompts/poker_action_selection_standard.txt"]
standard = (root / "agents/prompts/poker_action_selection_standard.txt").read_text().strip()
game_requests = [r for r in requests.values() if json.loads(r["user"])["observation"]["phase"] == "dialogue"]
assert len(game_requests) == 16
assert all(standard in r["system"] for r in game_requests)
assert all(json.loads(r["user"])["force_bet"] is False for r in game_requests)
events = read_rows("events.jsonl")
bets = [r for r in events if r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A"]
assert all(not r["event"].get("automatic", False) for r in bets)
assert not any(r["event"]["type"] == "AgentFailed" for r in events)
actions = Counter(r["event"]["action"] for r in bets)
assert actions == Counter(check=4, call=2)
print("Run:", run_id, "valid", len(trials), "model", manifest["model"], "effort=low; standard prompt verified")
print("Duration:", round(manifest["total_duration_s"], 3), "seconds")
print("Requests/responses:", len(requests), len(responses), "game/summary:", len(game_requests), len(responses)-len(game_requests))
print("Tokens:", {k: sum(r["usage"][k] for r in responses) for k in ("input_tokens", "output_tokens", "tokens")})
model_times = [r["duration_s"] for r in read_rows("timings.jsonl") if r["kind"] == "model"]
print("Mean model wall time:", round(sum(model_times)/len(model_times), 3))
print("Actual A actions:", dict(actions))
for t in sorted(trials, key=lambda t: t["trial_id"]):
    seq = [r["event"]["action"] for r in bets if r["trial_id"] == t["trial_id"]]
    print(t["trial_id"], seq, "winner", t["winner"], "delta", t["chip_delta"])
for old_id in ("20261021T065748-ed168d74", "20261021T081635-76bb44b5"):
    old_path = root / "experiments/results" / old_id
    old_manifest = json.loads((old_path / "manifest.json").read_text())
    assert old_manifest["source_hashes"]["agents/prompts/poker_action_selection.txt"] == manifest["source_hashes"]["agents/prompts/poker_action_selection.txt"]
    old_trials = {r["trial_id"]: r for r in map(json.loads, (old_path / "trials.jsonl").read_text().splitlines())}
    assert all(old_trials[t["trial_id"]]["input_hash"] == t["input_hash"] for t in trials)
    print("Matched input hashes + action template:", old_id, "4/4")
# Audit explicit output notes, not hidden reasoning. Retrieved records differ between runs.
request = next(r for r in game_requests if r["trial_id"] == "s3-C-01-both" and json.loads(r["user"])["observation"]["event_id"] == 3)
user = json.loads(request["user"])
matches = user["past_experiences"]["matches"]
assert len(matches) == 3
assert all(m["metadata"]["player_id"] == "C" and "네 카드는 10 이하야." in m["text"] for m in matches)
ranks = [m["round_context"]["result"]["revealed_cards"]["A"]["rank"] for m in matches]
assert ranks == [4, 5, 6]
assert all(rank <= 10 for rank in ranks)
response = next(r for r in responses if r["call_id"] == request["call_id"])
print("C/both event3 historical A ranks:", ranks, "claim truth 3/3")
print("Explicit note:", json.loads(response["content"])["memory_updates"]["round"])
print("C final memo:", next(t for t in trials if t["trial_id"] == "s3-C-01-both")["saved_memos"]["long"])
print("Caution: no explicit 3-case recount in output; seeded aggregate memory also supplied. This is not a proven retrieval-only reasoning gain.")
print("Caution: 4 independent rounds, no behavior difference control vs both; no win-rate conclusion. Japanese public speech persists in B/control.")


Run: 20261021T083322-7fc60706 valid 4 model gpt-6.1-sol effort=low; standard prompt verified
Duration: 97.47 seconds
Requests/responses: 20 20 game/summary: 16 4
Tokens: {'input_tokens': 47759, 'output_tokens': 2683, 'tokens': 50442}
Mean model wall time: 4.508
Actual A actions: {'check': 4, 'call': 2}
s3-B-01-both ['check', 'call'] winner A delta 6
s3-B-01-control ['check', 'call'] winner A delta 6
s3-C-01-both ['check'] winner A delta 2
s3-C-01-control ['check'] winner A delta 2
Matched input hashes + action template: 20261021T065748-ed168d74 4/4
Matched input hashes + action template: 20261021T081635-76bb44b5 4/4
C/both event3 historical A ranks: [4, 5, 6] claim truth 3/3
Explicit note: C의 관찰 카드는 9C이고 내 카드는 미확인이다. C는 내 카드가 10 이하라고 주장했으며 과거 일치 기록을 참고하되 확정하지 않고, 추가 지출 없는 체크를 잠정 선택한다.
C final memo: C의 'A 카드는 10 이하' 주장은 기존 공개 자료 14건과 이번 10H 공개 결과를 합쳐 15건 일치·0건 불일치로 관찰됐다. 기존 레이즈 4회·폴드 0회 기록은 보존하되, 이 일치 기록만으로 항상 정직하거나 베팅이 강하다고 확정하지 않는다.
Caution: no explicit 3-case recount in output; seede